# Zerobus Ingest — One Click Demo

Clone this notebook, then click Run All

## Auto configure [Public Demo](https://docs.databricks.com/aws/en/ingestion/zerobus-ingest)

- ZeroBus endpoint and workspace URL — [Get your workspace URL and ZeroBus Ingest endpoint](https://docs.databricks.com/aws/en/ingestion/zerobus-ingest#get-your-workspace-url-and-zerobus-ingest-endpoint)
- Service Principal - [Create a service principal and grant permissions](https://docs.databricks.com/aws/en/ingestion/zerobus-ingest#create-a-service-principal-and-grant-permissions)
- OATH SECRET - [OAuth secret](https://docs.databricks.com/aws/en/dev-tools/auth/oauth-m2m)
- Grant use, select, modify permission on Catalog, Schema, Table

## Normal table ingestion

Create 10 rows of the following
```
    for i in range(10):
        record_dict = {
            "device_name": f"sensor-{i}",
            "temp": 20 + i % 15,
            "humidity": 50 + i % 40
        }
        last_offset = stream.ingest_record_offset(record_dict)
```

In [1]:
%pip install --quiet databricks-zerobus-ingest-sdk

Note: you may need to restart the kernel to use updated packages.


In [2]:
# config for reuse 

_config = {
  "ZEROBUS_TABLE_NAME": "air_quality",
  "ZEORBUS_SCHEMA": "",                 # will be your name if not populated
  "ZEORBUS_CATALOG": "",                # will be main if not populated

  # save if generated by this notebook
  "ZEROBUS_SERVICE_PRINCIPAL_NAME": "", # will be lfcdemo-zerobus-sp if not populated 
  "ZEROBUS_SERVICE_PRINCIPAL_ID": "",   # will be generated if not populated
  "ZEROBUS_APP_ID": "",                 # will be generated if not populated
  "ZEROBUS_OAUTH_SECRET": "",           # will be generated if not populated or invalid

  # if this notebook cannot figure it out
  "DATABRICKS_WORKSPACE_URL": "", # auto detect if not populated or incorrect
  "ZEROBUS_SERVER_ENDPOINT": "",  # auto detect if not populated or incorrect
}

# merge to _config saved config.json from the file
import json
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / ".git").exists() or (parent / "config.json").exists():
            return parent
    return start

def _load_config(path: Path, defaults: dict) -> tuple[dict, dict]:
    """Load config.json and merge non-blank values over defaults.

    Returns (merged, saved_snapshot) where saved_snapshot is the raw disk
    content — used at the end of the notebook to detect changes.
    """
    saved = json.loads(path.read_text()) if path.exists() else {}
    # Notebook _config wins when non-blank; disk config.json fills in blank values.
    # Keys on disk that are not in defaults are included as-is.
    merged = {k: v if v else saved.get(k, v) for k, v in defaults.items()} | {
        k: v for k, v in saved.items() if k not in defaults
    }
    return merged, dict(saved)

def _save_config(path: Path, updates: dict) -> None:
    data = json.loads(path.read_text()) if path.exists() else {}
    data.update(updates)
    path.write_text(json.dumps(data, indent=2) + "\n")
    print(f"Saved {list(updates.keys())} to {path}")

def _save_config_if_changed(path: Path, current: dict, original: dict) -> None:
    """Save only the keys that changed relative to the disk snapshot."""
    changed = {k: v for k, v in current.items() if original.get(k) != v}
    if changed:
        _save_config(path, changed)
    else:
        print("config.json unchanged — nothing to save")

_repo_root         = _find_repo_root(Path(__file__).parent if "__file__" in dir() else Path.cwd())
_config, _config_original = _load_config(_repo_root / "config.json", _config)

In [3]:
from databricks.sdk import WorkspaceClient
from zerobus.sdk.sync import ZerobusSdk
from zerobus.sdk.shared import RecordType, StreamConfigurationOptions, TableProperties, HeadersProvider
import re, time
from databricks.sdk.errors import NotFound
_w = WorkspaceClient()   # used for SP auto-creation and PAT fallback

username = re.sub(r"[^a-z0-9]", "_", _w.current_user.me().user_name.split("@")[0])

In [4]:
# auto determine SERVER_ENDPOINT, DATABRICKS_WORKSPACE_URL

def get_endpoint_workspace_url(SERVER_ENDPOINT, DATABRICKS_WORKSPACE_URL):
    generated=False
    _url_mismatch = DATABRICKS_WORKSPACE_URL and DATABRICKS_WORKSPACE_URL != _w.config.host
    if _url_mismatch:
        print(f"Warning: DATABRICKS_WORKSPACE_URL={DATABRICKS_WORKSPACE_URL!r} does not match "
            f"WorkspaceClient host={_w.config.host!r}. Recalculating both values.")
        SERVER_ENDPOINT = ""
        DATABRICKS_WORKSPACE_URL = ""

    if not SERVER_ENDPOINT or not DATABRICKS_WORKSPACE_URL:
        if not DATABRICKS_WORKSPACE_URL:
            DATABRICKS_WORKSPACE_URL = _w.config.host
        if not SERVER_ENDPOINT:
            _workspace_id = _w.get_workspace_id()
            if "azuredatabricks.net" in DATABRICKS_WORKSPACE_URL:
                _region = spark.sql("SELECT current_metastore()").collect()[0][0].split(":")[1]
            else:
                _region = _w.clusters.list_zones().default_zone[:-1]     # AWS: strip trailing AZ letter
            _domain = "azuredatabricks.net" if "azuredatabricks.net" in DATABRICKS_WORKSPACE_URL else "cloud.databricks.com"
            SERVER_ENDPOINT = f"https://{_workspace_id}.zerobus.{_region}.{_domain}"
        generated=True
        print(f"Auto-derived SERVER_ENDPOINT={SERVER_ENDPOINT}")
        print(f"Auto-derived DATABRICKS_WORKSPACE_URL={DATABRICKS_WORKSPACE_URL}")

    return SERVER_ENDPOINT, DATABRICKS_WORKSPACE_URL, generated

_config["ZEROBUS_SERVER_ENDPOINT"], _config["DATABRICKS_WORKSPACE_URL"], generated = get_endpoint_workspace_url(
    _config.get("ZEROBUS_SERVER_ENDPOINT",""), 
    _config.get("DATABRICKS_WORKSPACE_URL",""))
print(f"{_config['ZEROBUS_SERVER_ENDPOINT']=}")
print(f"{_config['DATABRICKS_WORKSPACE_URL']=}")

Auto-derived SERVER_ENDPOINT=https://1444828305810485.zerobus.us-west-2.cloud.databricks.com
Auto-derived DATABRICKS_WORKSPACE_URL=https://e2-demo-field-eng.cloud.databricks.com
_config['ZEROBUS_SERVER_ENDPOINT']='https://1444828305810485.zerobus.us-west-2.cloud.databricks.com'
_config['DATABRICKS_WORKSPACE_URL']='https://e2-demo-field-eng.cloud.databricks.com'


In [5]:
# create service prinicpal if not exists

# to delete 
# databricks service-principals list --filter 'displayName co "robert" and displayName co "zerobus"' --output json \
#  | jq -r '["NAME","SP_ID","APP_ID"], (.[] | [.displayName, (.id | tostring), (.applicationId | tostring)]) | @tsv' \
#  | column -t
# databricks service-principals delete <sp_id>

def _ensure_sp() -> None:
    """Ensure a ZeroBus service principal exists.

    - If ZEROBUS_SERVICE_PRINCIPAL_ID is blank → create a new SP.
    - If it is set but the SP no longer exists → create a replacement.
    - Backfills ZEROBUS_APP_ID from the SP if missing.
    - Saves any changes to config.json.
    """
    sp_id = _config.get("ZEROBUS_SERVICE_PRINCIPAL_ID", "")
    _slug = re.sub(r"[^a-z0-9]", "_", _w.current_user.me().user_name.split("@")[0])

    def _create_sp():
        _name = _config["ZEROBUS_SERVICE_PRINCIPAL_NAME"] if _config["ZEROBUS_SERVICE_PRINCIPAL_NAME"] else f"lfcdemo_zerobus_sp"
        _config["ZEROBUS_SERVICE_PRINCIPAL_NAME"] = _name
        _sp = _w.service_principals.create(display_name=_name)
        _config["ZEROBUS_SERVICE_PRINCIPAL_ID"] = str(_sp.id)
        _config["ZEROBUS_APP_ID"]               = str(_sp.application_id)
        print(f"Created SP '{_name}' id={_sp.id} APP_ID={_config['ZEROBUS_APP_ID']}")

    if not sp_id:
        print("ZEROBUS_SERVICE_PRINCIPAL_ID not set — creating service principal")
        _create_sp()
    else:
        try:
            _sp = _w.service_principals.get(sp_id)
            print(f"SP exists: '{_sp.display_name}'  id={sp_id}")
            if not _config.get("ZEROBUS_APP_ID"):
                _config["ZEROBUS_APP_ID"] = str(_sp.application_id)
                _save_config(_repo_root / "config.json", {"ZEROBUS_APP_ID": _config["ZEROBUS_APP_ID"]})
                print(f"Backfilled ZEROBUS_APP_ID={_config['ZEROBUS_APP_ID']}")
        except NotFound:
            print(f"SP id={sp_id} not found — creating a replacement")
            _create_sp()

_ensure_sp()
print(f"ZEROBUS_SERVICE_PRINCIPAL_ID = {_config['ZEROBUS_SERVICE_PRINCIPAL_ID']}")
print(f"ZEROBUS_APP_ID               = {_config['ZEROBUS_APP_ID']}")


ZEROBUS_SERVICE_PRINCIPAL_ID not set — creating service principal


NotFound: Endpoint not supported

In [ ]:
# (re)create if not valid CLIENT_SECRET

def _credentials_valid(client_id: str, client_secret: str, DATABRICKS_WORKSPACE_URL) -> bool:
    """Test credentials by calling the OIDC token endpoint directly.

    WorkspaceClient(client_id=, client_secret=) silently falls back to
    ~/.databrickscfg, so a bogus secret would still pass. Calling the token
    endpoint with requests bypasses that fallback and actually validates the
    secret.
    """
    if not client_id or not client_secret:
        return False
    try:
        import requests as _req
        _resp = _req.post(
            f"{DATABRICKS_WORKSPACE_URL.rstrip('/')}/oidc/v1/token",
            data={
                "grant_type": "client_credentials",
                "client_id": client_id,
                "client_secret": client_secret,
                "scope": "all-apis",
            },
            timeout=10,
        )
        if not _resp.ok:
            print(f"Exception: {_resp.status_code} {_resp.text}")
        return _resp.ok
    except Exception as ex:
        print(f"Exception: {ex}")
        return False
      
client_and_secret_valid=_credentials_valid(
    _config['ZEROBUS_APP_ID'], 
    _config['ZEROBUS_OAUTH_SECRET'], 
    _config['DATABRICKS_WORKSPACE_URL'])

if client_and_secret_valid:
    print("client (app) secret is valid")
else:
    print("client (app) secret is not valid")
    _sp_id = _config["ZEROBUS_SERVICE_PRINCIPAL_ID"]
    _secret_obj = None
    try:
        _secret_obj = _w.service_principal_secrets_proxy.create(service_principal_id=_sp_id)
    except AttributeError:
        try:
            # SDK < 0.28: ServicePrincipalSecretsAPI not on WorkspaceClient directly
            from databricks.sdk import ServicePrincipalSecretsAPI
            _secret_obj = ServicePrincipalSecretsAPI(_w.api_client).create(service_principal_id=_sp_id)
        except Exception:
            # Final fallback: workspace proxy REST endpoint
            _resp = _w.api_client.do(
                'POST', f'/api/2.0/accounts/servicePrincipals/{_sp_id}/credentials/secrets'
            )
            _config["ZEROBUS_OAUTH_SECRET"] = _resp["secret"]

    if _secret_obj is not None:
        _config["ZEROBUS_OAUTH_SECRET"] = _secret_obj.secret
    _save_config(_repo_root / "config.json", {"ZEROBUS_OAUTH_SECRET": _config["ZEROBUS_OAUTH_SECRET"]})
    print(f"Created new ZEROBUS_OAUTH_SECRET and saved to config.json")

In [ ]:
# create target schema, table

fq_table_name_parts = _config['ZEROBUS_TABLE_NAME'].split('.')
table_catalog = "main" if len(fq_table_name_parts) <= 2 else fq_table_name_parts.pop(0)
table_schema = username if len(fq_table_name_parts) < 2 else fq_table_name_parts.pop(0)
table_name = fq_table_name_parts.pop(0) if len(fq_table_name_parts) > 0 else "air_quality"
fq_table_name=f"{table_catalog}.{table_schema}.{table_name}"

if not table_name: raise Exception("table not specfied")

print(f"{table_catalog=} {table_schema=} {table_name=}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {table_catalog}.{table_schema}")

# Drop and recreate if the schema has been corrupted by a previous Scenario 2 run.
_expected = {"device_name", "temp", "humidity"}
try:
    _actual = {f.name for f in spark.table(fq_table_name).schema}
    if not _expected.issubset(_actual):
        print(f"Schema mismatch (found {_actual}), dropping and recreating...")
        spark.sql(f"DROP TABLE IF EXISTS {fq_table_name}")
except Exception:
    pass  # table doesn't exist yet — CREATE below will handle it

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {fq_table_name} (
  device_name STRING,
  temp        INT,
  humidity    INT
)
""")


In [ ]:
# grant use on catalog, schema, select modify on target table

def _has_grant(principal, action, securable_type, securable_name):
    """Return True if `principal` already has `action` on the given securable."""
    rows = spark.sql(f"SHOW GRANTS `{principal}` ON {securable_type} {securable_name}").collect()
    return any(r["ActionType"] == action for r in rows)

# Only grant if the principal doesn't already have the privilege
if not _has_grant(_config['ZEROBUS_APP_ID'], "USE CATALOG", "CATALOG", table_catalog):
    spark.sql(f"GRANT USE CATALOG ON CATALOG {table_catalog} TO `{_config['ZEROBUS_APP_ID']}`")
    print(f"WARNING: Granted USE CATALOG on {table_catalog}")
else:
    print(f"USE CATALOG on {table_catalog} already granted")

if not _has_grant(_config['ZEROBUS_APP_ID'], "USE SCHEMA", "SCHEMA", f"{table_catalog}.{table_schema}"):
    spark.sql(f"GRANT USE SCHEMA ON SCHEMA {table_catalog}.{table_schema} TO `{_config['ZEROBUS_APP_ID']}`")
    print(f"WARNING: Granted USE SCHEMA on {table_catalog}.{table_schema}")
else:
    print(f"USE SCHEMA on {table_catalog}.{table_schema} already granted")

for action in ["MODIFY", "SELECT"]:
    if not _has_grant(_config['ZEROBUS_APP_ID'], action, "TABLE", fq_table_name):
        spark.sql(f"GRANT {action} ON TABLE {fq_table_name} TO `{_config['ZEROBUS_APP_ID']}`")
        print(f"WARNING: Granted {action} on {fq_table_name}")
    else:
        print(f"{action} on {fq_table_name} already granted")

In [ ]:
# url of the table, table storage, and reject rows

_table_url = f"{_config['DATABRICKS_WORKSPACE_URL'].rstrip('/')}/explore/data/{table_catalog}/{table_schema}/{table_name}"
print(f"Table: {_table_url}")

_detail = spark.sql(f"DESCRIBE DETAIL {fq_table_name}").collect()[0]
_location = _detail["location"]
_rejected_path = f"{_location}/_zerobus/table_rejected_parquets/"

print(f"Table storage : {_location}")
print(f"Rejected rows : {_rejected_path}")

In [ ]:
# Normal ingest — 10 rows of valid data matching the table schema

sdk = ZerobusSdk(
    _config['ZEROBUS_SERVER_ENDPOINT'],
    _config['DATABRICKS_WORKSPACE_URL']
)

table_properties = TableProperties(fq_table_name)
options = StreamConfigurationOptions(record_type=RecordType.JSON)
stream = sdk.create_stream(_config['ZEROBUS_APP_ID'], _config['ZEROBUS_OAUTH_SECRET'], table_properties, options)

try:
    last_offset = None
    for i in range(10):
        record_dict = {
            "device_name": f"sensor-{i}",
            "temp": 20 + i % 15,
            "humidity": 50 + i % 40
        }
        last_offset = stream.ingest_record_offset(record_dict)

    # Wait once for the final offset — all prior offsets are guaranteed committed too
    if last_offset is not None:
        stream.wait_for_offset(last_offset)
finally:
    stream.close()


In [ ]:
# min max on device_name, count

display(
spark.sql(f"select min(device_name), max(device_name), count(*) from {fq_table_name}")
)


In [ ]:
# row count by device id

display(
spark.sql(f"""
SELECT device_name, count(*) AS row_count
FROM {fq_table_name}
GROUP BY device_name
HAVING count(*) > 1
ORDER BY CAST(regexp_extract(device_name, '(\\\\d+)$', 1) AS INT), row_count DESC
""")
)

In [ ]:
# show num of records processed each time we called zeorbus 
# there will be some delays

try:
    display(spark.sql(f"""
    SELECT commit_time, table_name, committed_records, errors
    FROM system.lakeflow.zerobus_ingest
    WHERE table_name = '{fq_table_name}'
    ORDER BY commit_time DESC
    LIMIT 20
    """))
except Exception as e:
    if "TABLE_OR_VIEW_NOT_FOUND" in str(e):
        print("system.lakeflow.zerobus_ingest is not available in this workspace.")
        print("Ask a workspace admin to enable the ZeroBus Ingest system table.")
    else:
        raise

In [ ]:
# save config if anything changed

_save_config_if_changed(_repo_root / "config.json", _config, _config_original)